## Download and use a pre-trained model from Hugging Face

Prerequisites:
- Install the `transformers` library if you haven't already:
```bash
    pip install transformers
   ```
- Obtain an API token from Hugging Face and set it up in your environment:
```bash
    huggingface-cli login
   ```
- This will allow you to download models from the Hugging Face Model Hub.

In [17]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-hf",
                                             torch_dtype="auto",
                                             device_map="auto",
                                             cache_dir="./cache")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")

model_input = tokenizer("Hello, my name is Gerald and I am an AI engineer.", return_tensors="pt").to(model.device)

Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.99s/it]
Some parameters are on the meta device because they were offloaded to the disk.


In [18]:
generated_ids = model.generate(**model_input, max_length=30)
tokenizer.batch_decode(generated_ids)[0]

'<s> Hello, my name is Gerald and I am an AI engineer. I am currently a PhD student in the field of Machine Learning.'

In [19]:
model_input = tokenizer("What is AI ?", return_tensors="pt").to(model.device)
generated_ids = model.generate(**model_input, max_length=100)
tokenizer.batch_decode(generated_ids)[0]

'<s> What is AI ?\nA.I. stands for Artificial Intelligence. It is the science of making computers think like humans. It is also the art of making computers do things that normally require human intelligence.\nHow is AI different from Machine Learning ?\nMachine learning is a subset of AI. Machine learning is the process of teaching machines to learn and improve from experience.\nWhat is Deep Learning ?\nDeep learning is a type of machine learning that enables computers'

> Here MPS used longer time to generate the text compared to CPU.

In [16]:
model_input = tokenizer("What is AI ?", return_tensors="pt")
generated_ids = model.generate(**model_input, max_length=100)
tokenizer.batch_decode(generated_ids)[0]

/opt/anaconda3/envs/NLP/lib/python3.10/site-packages/transformers/generation/utils.py:2495: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on mps. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('mps') before running `.generate()`.
  warnings.warn(


'<s> What is AI ?\nWhat is AI ?AI (Artificial Intelligence) is a branch of computer science that is concerned with the design and development of intelligent systems. AI systems are designed to mimic human intelligence and perform tasks that would normally require human intelligence, such as decision-making, problem-solving, and pattern recognition.\nAI systems are powered by machine learning algorithms that enable them to learn from data and improve their performance over time. They'

All of this is very slow, so using vanilla transformers is not recommended for inference. Instead use vllm or trl for inference.

In [21]:
from vllm import LLM, SamplingParams
llm = LLM(model="meta-llama/Llama-2-7b-hf", )
sampling_params = SamplingParams(max_tokens=1000, temperature=0.7, top_p=0.95)
response = llm.generate("What is AI ?", sampling_params=sampling_params)
print(response[0].outputs[0].text)

INFO 07-03 12:07:17 [config.py:823] This model supports multiple tasks: {'reward', 'classify', 'embed', 'score', 'generate'}. Defaulting to 'generate'.
INFO 07-03 12:07:17 [config.py:1980] Disabled the custom all-reduce kernel because it is not supported on current platform.
WARNING 07-03 12:07:17 [cpu.py:135] Environment variable VLLM_CPU_KVCACHE_SPACE (GiB) for CPU backend is not set, using 4 by default.
INFO 07-03 12:07:17 [llm_engine.py:230] Initializing a V0 LLM engine (v0.9.1) with config: model='meta-llama/Llama-2-7b-hf', speculative_config=None, tokenizer='meta-llama/Llama-2-7b-hf', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=True, quantization=None, enforce_eager=True, kv_cache_dtype=auto,  device_config=cpu, decoding_config=Decoding

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:30<00:30, 30.71s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:41<00:00, 18.77s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:41<00:00, 20.56s/it]


INFO 07-03 12:08:00 [default_loader.py:272] Loading weights took 41.14 seconds
INFO 07-03 12:08:00 [executor_base.py:113] # cpu blocks: 512, # CPU blocks: 0
INFO 07-03 12:08:00 [executor_base.py:118] Maximum concurrency for 4096 tokens per request: 2.00x


INFO 07-03 12:08:01 [llm_engine.py:428] init engine (profile, create kv cache, warmup model) took 0.40 seconds


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


WARNING 07-03 12:08:05 [cpu.py:243] Pin memory is not supported on CPU.


Processed prompts: 100%|██████████| 1/1 [03:39<00:00, 219.20s/it, est. speed input: 0.03 toks/s, output: 4.56 toks/s]


What is AI ? What are the applications of AI ?
The term artificial intelligence (AI) is widely used to describe many different things. In the broadest sense, it can be described as the development of computer systems that can perform tasks that normally require human intelligence.
AI is a broad field of study that includes many different subfields. These include machine learning, natural language processing, and computer vision. AI has many potential applications, including autonomous vehicles, medical diagnosis, and fraud detection.
AI is a branch of computer science that deals with the simulation of human intelligence processes by machines, especially computer systems. AI includes techniques such as machine learning, natural language processing, and speech recognition.
AI is used in many different fields, including business, finance, healthcare, and manufacturing. AI can help businesses automate tasks, improve decision-making, and optimize operations. AI can also help healthcare pro

In [22]:
response = llm.generate("How did God create the world ?", sampling_params=sampling_params)
print(response[0].outputs[0].text)

Processed prompts: 100%|██████████| 1/1 [03:37<00:00, 217.72s/it, est. speed input: 0.04 toks/s, output: 4.59 toks/s]


Scientists have only one way to know how God created the universe. This is through their scientific theories.
Scientists have a theory called the Big Bang Theory. According to this theory, the universe was created by a gigantic explosion.
In the beginning, there was only a tiny speck of matter which was so tiny that it could not even be seen. Then, at a certain moment, this speck of matter exploded and the universe was created.
This theory is not true. In fact, it is impossible for a speck of matter to explode and create the universe.
The universe was created by God.
The Bible says that the universe was created by God.
In the beginning, God created the heavens and the earth.
The earth was without form, and void; and darkness was on the face of the deep.
And the Spirit of God moved on the face of the waters.
And God said, Let there be light: and there was light.
And God called the light Day, and the darkness he called Night.
And God said, Let the waters under the heaven be gathered tog

In [23]:
response = llm.generate("who is the third president of Tanzania", sampling_params=sampling_params)
print(response[0].outputs[0].text)

Processed prompts: 100%|██████████| 1/1 [04:22<00:00, 262.65s/it, est. speed input: 0.04 toks/s, output: 3.81 toks/s]

?
Where is the presidential palace in Tanzania?
Who is the first president of Tanzania?
Who is the President of Tanzania 2021?
How much is the President of Tanzania salary?
Who was the first president of East Africa?
Who is the first president of Tanzania?
Who is the president of Tanzania 2020?
Who is the current president of Tanzania?
What is the president of Tanzania salary?
How much is the President of Tanzania salary?
Who is the first president of East Africa?
Who is the President of Tanzania 2020?
What is the salary of the President of Tanzania?
How much is the President of Tanzania salary?
Who is the President of Tanzania?
Who is the President of Tanzania now?
Who is the president of Tanzania?
Who is the President of Tanzania 2020?
Who is the current president of Tanzania?
How much is the salary of the President of Tanzania?
How much is the President of Tanzania salary?
Who is the President of Tanzania 2021?
Who is the President of Tanzania?
How much is the salary of the Presiden